In [1]:
import torch
from PIL import Image
from transformers.utils.import_utils import is_flash_attn_2_available

from colpali_engine.models import ColQwen2, ColQwen2Processor

model_name = "vidore/colqwen2-v1.0"

model = ColQwen2.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",  # or "mps" if on Apple Silicon
    attn_implementation="flash_attention_2" if is_flash_attn_2_available() else None,
).eval()
processor = ColQwen2Processor.from_pretrained(model_name)

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [31]:
queries = [
    "emo rap song with all blue album art",
]

# Process the inputs
inputs_text = processor.process_queries(queries).to(model.device)

# Forward pass
with torch.no_grad():
    query_embeddings = model(**inputs_text).cpu().float().numpy()


In [14]:
from qdrant_client import QdrantClient, models

qdrant = QdrantClient("http://localhost:6333")
collection_name = "colqwen2_embeddings"

def reranking_search_batch(query_batch,
                           collection_name,
                           search_limit=20,
                           prefetch_limit=200):
    search_queries = [
      models.QueryRequest(
          query=query,
          prefetch=[
              models.Prefetch(
                  query=query,
                  limit=prefetch_limit,
                  using="mean_pooling_columns"
              ),
              models.Prefetch(
                  query=query,
                  limit=prefetch_limit,
                  using="mean_pooling_rows"
              ),
          ],
          limit=search_limit,
          with_payload=True,
          with_vector=False,
          using="original"
      ) for query in query_batch
    ]
    return qdrant.query_batch_points(
        collection_name=collection_name,
        requests=search_queries
    )

In [32]:
for point in reranking_search_batch(query_embeddings, collection_name)[0].points:
    print(f"name - {point.payload['track_name']}, score - {point.score}")

name - Tied, score - 16.038113
name - FEELING, score - 15.927172
name - My Life Is Tragic, score - 15.874223
name - when it wasn't strange, score - 15.833367
name - catch you a star, score - 15.759319
name - TOOTHPASTE - DEDLINE PHONK REMIX, score - 15.603681
name - highschool (feat. convolk) - remix, score - 15.513904
name - West End, score - 15.281785
name - erode, score - 15.165274
name - AWKWARD CAR DRIVE, score - 15.15173
name - KENSHI, score - 13.07843
name - Ammo, score - 13.071216
name - Xanman, score - 12.851422
name - Don’t Blink Or Stare (feat. Bloodhound Lil Jeff & CEO Trayle), score - 12.82369
name - The Adventures of Moon Man & Slim Shady (with Eminem), score - 12.8108015
name - Techno Phonk, score - 12.642302
name - tv off (feat. lefty gunplay), score - 12.586115
name - Task Force, score - 12.57396
name - Illusion, score - 12.449386
name - #BrooklynBloodPop!, score - 12.382728


In [6]:
reranking_search_batch(query_embeddings, collection_name)

[QueryResponse(points=[ScoredPoint(id='34de3c0a-18bc-4f66-82c0-ebfc529bdda8', version=2, score=11.147018, payload={'track_id': '5bFuHlXKw66Uu2cHKn5bf8', 'track_name': 'Alive'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='9cf4a17a-eb2d-4f26-86f4-0dcdf46bb7e5', version=0, score=10.882036, payload={'track_id': '2gHJ7wDPoLDnYp5poSMQ82', 'track_name': 'Dark Sphere'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='25c03491-3cdc-467a-8e48-28e7e33e2776', version=2, score=9.817109, payload={'track_id': '2fQgFJWkIPGh9q1uKaYSI5', 'track_name': 'Tied'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='93796e60-b190-4cbf-ac04-c7fcb1e1b98e', version=4, score=9.621855, payload={'track_id': '4W2PK3gIBR0coRIyOvKeGS', 'track_name': "when it wasn't strange"}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='f8e60671-20d4-4a54-a617-4118a407abdc', version=9, score=9.524263, payload={'track_id': '6RZBBle1IHtR2TGIHRRhCI', 'track_name': 'TOOTH

In [2]:
image_embeddings.shape

torch.Size([2, 36, 128])